# M15 Notebook B: SHAP explanations and LLM verification, full pool, seed 42

Retrains E4 (`e4_added`), explains it with TreeSHAP and runs the gated LLM verifier (Gemma 2 2B) on a smaller sample.

**What it runs:**
- the M2 partition;
- M10: refits `e4_added` and saves each window's models, then explains them with TreeSHAP;
- M11: refits `e4_added` with the gate's capture hook, then runs the verifier on 100 rows per client and window, 10 of them always-on. M11 itself used 500 and 50.

**Deviation from the proposal (recorded in the README):** the proposal's Notebook B loads Notebook A's frozen models, but this one retrains them, as M10 and M11 always have. Each refit is checked against M9's committed scores.

**The check:** `scripts/m15_check.py --part B` passes if both refits are within ±0.005 macro-F1 of M9 in every window, and writes `reports/m15/full/CHECK_B.md`. The verifier's results on the smaller sample are reported, not tested.

**Attach as input:**
1. The processed CICIoMT2024 dataset (see `release/dataset/README.md`). Its SHA-256s are checked first.
2. *Add Input → Models →* **Gemma 2**, *Transformers*, **gemma-2-2b-it** (accept Google's licence on the model page first).

**Settings:** Internet *On*, Accelerator **GPU T4**.
**Run:** *Save Version → Save & Run All (Commit)*.

**Time:** about 5 h, estimated, not measured. That is two refits of about 80 min each, SHAP, and about a fifth of M11's LLM time. Kaggle counts the whole session against the GPU quota.

**Output:** `/kaggle/working/m15_b_seed42_outputs.zip`, rewritten after every step. It holds the M10 and M11 reports with every prompt and reply (`llm_log.jsonl`), and the run folders.


In [ ]:
SEED = 42   # the check compares against the committed seed-42 reports, so keep 42
import glob, hashlib, os, shlex, shutil, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
# bitsandbytes (4-bit Gemma) is not preinstalled
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m15_check.py"), "This clone predates M15. Restart the session so it clones the latest main."

# The attached data must be exactly the released processed dataset
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "No data_manifest.json under /kaggle/input: Add Input -> the processed CICIoMT2024 dataset."
want = {name: sha for sha, name in (line.split() for line in open("release/dataset/SHA256SUMS"))}
for name, sha in want.items():
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} is not under /kaggle/input: attach the processed CICIoMT2024 dataset"
    h = hashlib.sha256()
    with open(hits[0], "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    assert h.hexdigest() == sha, f"{name}: SHA-256 differs from release/dataset/SHA256SUMS, so this is not the released data"
    print("data ok:", name)


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
# The model: a Kaggle model input if attached, else Hugging Face (needs the HF_TOKEN secret)
found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "gemma-2-2b-it" in p.lower())
if found:
    MODEL = found[-1]
else:
    MODEL = "google/gemma-2-2b-it"
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("model:", MODEL)


In [ ]:
def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True).stdout

def pack():
    """Zip every report file this run changed or added (no checkpoints, no models) and the run folders."""
    stage = "/kaggle/working/m15_stage"
    shutil.rmtree(stage, ignore_errors=True)
    files = sorted(set(git("ls-files", "-m", "--", "reports").split()) | set(git("ls-files", "-o", "--", "reports").split()))
    files = [f for f in files if os.path.isfile(f) and "/checkpoints/" not in f]
    for f in files:
        os.makedirs(os.path.dirname(f"{stage}/{f}"), exist_ok=True)
        shutil.copy2(f, f"{stage}/{f}")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}", ignore=shutil.ignore_patterns("models"))
    os.makedirs(stage, exist_ok=True)
    shutil.make_archive(OUT, "zip", stage)
    shutil.rmtree(stage)
    print(f"zip written: {OUT}.zip ({len(files)} report files)", flush=True)

def step(cmd):
    print("=====", cmd, flush=True)
    rc = subprocess.run(shlex.split(cmd)).returncode
    print("exit code:", rc, flush=True)
    pack()
    return rc


In [ ]:
OUT = f"/kaggle/working/m15_b_seed{SEED}_outputs"
SET = f"--set resources.n_jobs=4 seed={SEED}"   # 4 threads, as in M9, so the refits can match M9's scores
LLM = f"llm.backend=hf llm.model_id={MODEL} llm.rows_per_client=100 llm.always_on_rows=10"
PY = "python -X faulthandler"
ARGS = f"--env kaggle --pool full {SET}"
STEPS = [
    f"{PY} scripts/m2_build_partitions.py --env kaggle --pools full",
    "git checkout -- reports/m2/full/*.png",
    f"{PY} scripts/m10_shap.py {ARGS} --only fit",
    f"{PY} scripts/m10_shap.py {ARGS} --only shap",
    f"{PY} scripts/m11_verify.py {ARGS} {LLM} --only fit",
    f"{PY} scripts/m11_verify.py {ARGS} {LLM} --only llm",
    "python scripts/m15_check.py --part B",
]
print("\n".join(STEPS))


In [ ]:
for c in STEPS:
    step(c)


In [ ]:
for f in ("reports/m15/full/CHECK_B.md", "reports/m11/full/VERIFY.md"):
    print(open(f, encoding="utf-8").read()[:6000] if os.path.exists(f) else f"{f}: not written, see the exit codes above")
